---
### 🐛 Found a bug or have a suggestion?
[Open a GitHub Issue](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues/new?template=bug_report.md)
*Search [existing issues](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues) first.*

### 💾 Save your own copy
> **File → Save a copy in Drive** — saves a personal editable copy to your Google Drive.
> Do this before making edits, otherwise changes are lost when the session ends.

# 🔀 Partial Least Squares (PLS)
**ISLP Chapter 6 · Pattern Recognition for the Rest of Us**

> PCR finds components that explain X. PLS finds components that explain both X **and Y** simultaneously — the direction that best predicts ridership, not just the direction that explains weather variation.

### Dataset: Capital Bikeshare — Washington DC (2011–2012)
Same dataset as PCR — enables direct method comparison.
**Business / public policy:** predict hourly bike demand for infrastructure planning.

### What you'll learn
- How PLS components differ from PCA components (NIPALS algorithm)
- Why PLS uses fewer components than PCR for the same accuracy
- VIP scores — which features matter most for predicting demand
- Head-to-head: OLS vs Ridge vs Lasso vs PCR vs PLS

### Time: ~40 minutes

In [ ]:
# ⚠️  Run this cell first — sets up data and imports
# Tip: Runtime → Run all  (runs everything top to bottom)

import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f8f9fa',
    'axes.grid': True, 'grid.alpha': 0.4,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 11
})

# ── Load Capital Bikeshare dataset (Washington DC, 2011-2012) ─────────────────
# Source: ISLP Chapter 7 / UCI ML Repository
# 8,645 hourly records — predict total bike rentals (bikers) from weather & time
# Key multicollinearity: temp & atemp (r = 0.99) — apparent temperature mirrors
# actual temperature, making OLS coefficients unstable

bikeshare = pd.read_csv('https://raw.githubusercontent.com/ladataanalytics/pattern-recognition-notebooks/main/data/Bikeshare.csv')
print(f'✓ Bikeshare: {bikeshare.shape}')

# Encode categoricals as numeric
bikeshare['mnth_num']    = pd.Categorical(bikeshare['mnth']).codes
bikeshare['weather_num'] = pd.Categorical(bikeshare['weathersit']).codes

# Features and target
features = ['temp','atemp','hum','windspeed',
            'hr','season','weekday','workingday',
            'holiday','mnth_num','weather_num']
target   = 'bikers'
p        = len(features)

X = bikeshare[features].values.astype(float)
y = bikeshare[target].values.astype(float)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

scaler  = StandardScaler().fit(X_tr)
X_tr_s  = scaler.transform(X_tr)
X_te_s  = scaler.transform(X_te)

print(f'  Features ({p}): {features}')
print(f'  Target: hourly bike rentals, mean={y.mean():.1f}, range={y.min():.0f}–{y.max():.0f}')
print(f'  Train: {X_tr_s.shape}  Test: {X_te_s.shape}')
print(f'  Python {sys.version.split()[0]} | pandas {pd.__version__}')
print('✓ Setup complete')


## 📐 Part 1 — PCR vs PLS: The Core Difference

Both compress X into M components before regressing. The difference is **what the components optimise**:

| | Component direction maximises… |
|---|---|
| **PCA / PCR** | Variance in **X** (weather patterns, regardless of ridership) |
| **PLS** | Covariance between **X and Y** (weather patterns that predict ridership) |

**For Bikeshare:** PCR's first component captures the temp/atemp axis (dominant X-variance). PLS's first component captures the direction in feature space that most directly predicts bike demand — which may combine temperature, hour, and weekday together.

In [ ]:
# ── Visualise PCA vs PLS components coloured by ridership ────────────────────
pca  = PCA(n_components=2).fit(X_tr_s)
pls2 = PLSRegression(n_components=2).fit(X_tr_s, y_tr)

Z_pca = pca.transform(X_tr_s)
Z_pls = pls2.transform(X_tr_s)[0]

# Subsample for scatter (8k points is too dense)
rng = np.random.default_rng(42)
idx = rng.choice(len(X_tr_s), size=1500, replace=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, Z, title, method in zip(
        axes,
        [Z_pca[idx], Z_pls[idx]],
        ['PCA (PCR) — components explain X variance\n(temp/atemp dominate)',
         'PLS — components explain X→Y covariance\n(oriented toward ridership)'],
        ['PCA', 'PLS']):
    sc = ax.scatter(Z[:,0], Z[:,1], c=y_tr[idx], cmap='YlOrRd',
                    alpha=0.5, s=15, edgecolors='none')
    plt.colorbar(sc, ax=ax, label='Hourly rentals')
    ax.set_xlabel(f'{method} Component 1')
    ax.set_ylabel(f'{method} Component 2')
    ax.set_title(title)

plt.suptitle('PCR vs PLS: Component 1 coloured by bike demand\n'
             'Better colour gradient = component explains demand better',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

r_pca = np.corrcoef(Z_pca[:,0], y_tr)[0,1]
r_pls = np.corrcoef(Z_pls[:,0], y_tr)[0,1]
print(f"Correlation of Component 1 with hourly rentals:")
print(f"  PCA Component 1: r = {r_pca:.3f}  (maximises X variance)")
print(f"  PLS Component 1: r = {r_pls:.3f}  (maximises X-Y covariance)")
print()
print("📌 PLS Component 1 is more correlated with Y — by design.")
print("   For demand forecasting, this means fewer components needed to")
print("   achieve the same predictive accuracy as PCR.")


## 🔬 Part 2 — Choosing M via Cross-Validation

In [ ]:
# ── PLS: cross-validate over M = 1 … p ──────────────────────────────────────
M_range  = range(1, p + 1)
cv_mses  = []
for M in M_range:
    pls  = PLSRegression(n_components=M)
    mse  = -cross_val_score(pls, X_tr_s, y_tr, cv=10,
                             scoring='neg_mean_squared_error').mean()
    cv_mses.append(mse)

best_M_pls   = list(M_range)[np.argmin(cv_mses)]
best_mse_pls = min(cv_mses)

# PCR for comparison (refit here)
pcr_cv = []
for M in M_range:
    pipe = Pipeline([('pca', PCA(n_components=M)), ('ols', LinearRegression())])
    mse  = -cross_val_score(pipe, X_tr_s, y_tr, cv=10,
                             scoring='neg_mean_squared_error').mean()
    pcr_cv.append(mse)
best_M_pcr   = list(M_range)[np.argmin(pcr_cv)]
ols_cv_mse   = -cross_val_score(LinearRegression(), X_tr_s, y_tr, cv=10,
                                  scoring='neg_mean_squared_error').mean()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(list(M_range), cv_mses,  'o-', color='#e85d20', lw=2,
        markersize=5, label='PLS 10-fold CV MSE')
ax.plot(list(M_range), pcr_cv,   's--', color='#1e5fa8', lw=2,
        markersize=5, label='PCR 10-fold CV MSE')
ax.axhline(ols_cv_mse, color='grey', lw=1.5, ls='--',
           label=f'OLS CV MSE = {ols_cv_mse:.1f}')
ax.axvline(best_M_pls, color='#e85d20', lw=2, ls=':',
           label=f'PLS best M = {best_M_pls}')
ax.axvline(best_M_pcr, color='#1e5fa8', lw=2, ls=':',
           label=f'PCR best M = {best_M_pcr}')
ax.set_xlabel('Number of Components M')
ax.set_ylabel('10-fold CV MSE')
ax.set_title('PLS vs PCR — CV MSE by Number of Components (Bikeshare)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

pls_best     = PLSRegression(n_components=best_M_pls).fit(X_tr_s, y_tr)
pcr_best     = Pipeline([('pca', PCA(n_components=best_M_pcr)),
                          ('ols', LinearRegression())]).fit(X_tr_s, y_tr)
pls_test_mse = mean_squared_error(y_te, pls_best.predict(X_te_s))
pcr_test_mse = mean_squared_error(y_te, pcr_best.predict(X_te_s))

print(f"PLS best M = {best_M_pls}  |  PCR best M = {best_M_pcr}")
print(f"PLS test RMSE = {pls_test_mse**0.5:.2f} rentals/hr")
print(f"PCR test RMSE = {pcr_test_mse**0.5:.2f} rentals/hr")


## 📊 Part 3 — The Grand Comparison

All five methods on the same Bikeshare train/test split.
Which method best predicts hourly bike demand?

In [ ]:
# ── Grand comparison: OLS, Ridge, Lasso, PCR, PLS ────────────────────────────
ols   = LinearRegression().fit(X_tr_s, y_tr)
ridge = RidgeCV(alphas=np.logspace(-3, 4, 100), cv=10).fit(X_tr_s, y_tr)
lasso = LassoCV(alphas=np.logspace(-4, 2, 100), cv=10,
                random_state=42, max_iter=5000).fit(X_tr_s, y_tr)

models = {
    'OLS':                                 ols,
    f'Ridge (λ={ridge.alpha_:.3f})':       ridge,
    f'Lasso ({np.sum(lasso.coef_!=0)} feat)': lasso,
    f'PCR   (M={best_M_pcr})':            pcr_best,
    f'PLS   (M={best_M_pls})':            pls_best,
}

print(f"{'Method':30s}  {'Test MSE':>9}  {'Test RMSE':>10}  {'rentals/hr':>12}")
print("-" * 68)
comparison = {}
for name, model in models.items():
    y_pred = model.predict(X_te_s).ravel()
    mse    = mean_squared_error(y_te, y_pred)
    comparison[name] = mse
    print(f"{name:30s}  {mse:>9.1f}  {mse**0.5:>10.2f}  ±{mse**0.5:.0f} rentals/hr")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
names  = list(comparison.keys())
mses   = list(comparison.values())
colors = ['#888888','#1e5fa8','#e85d20','#1a7a45','#c0392b']
bars = axes[0].barh(names, [m**0.5 for m in mses], color=colors,
                    edgecolor='white', height=0.6)
axes[0].axvline(min(m**0.5 for m in mses), color='gold', lw=2, ls='--', label='Best')
axes[0].set_xlabel('Test RMSE (hourly rentals)')
axes[0].set_title('Prediction Accuracy — Bikeshare Demand')
axes[0].legend(fontsize=9)
for bar, v in zip(bars, [m**0.5 for m in mses]):
    axes[0].text(v + 0.3, bar.get_y()+bar.get_height()/2,
                 f'{v:.1f}', va='center', fontsize=9)

# Actual vs predicted for best model
best_name = min(comparison, key=comparison.get)
best_model = list(models.values())[list(models.keys()).index(best_name)]
y_pred_best = best_model.predict(X_te_s).ravel()
sample = rng.choice(len(y_te), 300, replace=False)
axes[1].scatter(y_te[sample], y_pred_best[sample], alpha=0.4, s=15, color='#1e5fa8')
lims = [min(y_te.min(), y_pred_best.min()), max(y_te.max(), y_pred_best.max())]
axes[1].plot(lims, lims, 'r--', lw=1.5, label='Perfect prediction')
axes[1].set_xlabel('Actual rentals per hour')
axes[1].set_ylabel('Predicted rentals per hour')
axes[1].set_title(f'Actual vs Predicted — {best_name}\n(300 test samples)')
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.show()

best_method = min(comparison, key=comparison.get)
print(f"\n📌 Best on this split: {best_method}")
print("   For transportation planning, RMSE in original units is most actionable:")
print(f"   ±{min(mses)**0.5:.0f} rentals/hr error → plan dock capacity with that buffer.")


## ✅ Part 4 — VIP Scores: Which Features Drive Demand?

**Variable Importance in Projection (VIP)** measures each feature's contribution to the PLS model across all components. VIP > 1 means the feature contributes more than average — use this for feature selection and stakeholder communication.

In [ ]:
# ── VIP scores ────────────────────────────────────────────────────────────────
W = pls_best.x_weights_
T = pls_best.x_scores_
Q = pls_best.y_loadings_

ss       = (T**2).sum(axis=0) * (Q**2).sum(axis=0)
vip_raw  = np.sqrt(p * (W**2 @ ss) / ss.sum())
vip_df   = pd.DataFrame({'Feature': features, 'VIP': vip_raw})             .sort_values('VIP', ascending=False).reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# VIP bar chart
colors_vip = ['#1a7a45' if v >= 1.0 else '#1e5fa8' if v >= 0.8 else '#aaa'
               for v in vip_df['VIP']]
axes[0].barh(vip_df['Feature'], vip_df['VIP'], color=colors_vip, edgecolor='white')
axes[0].axvline(1.0, color='#e85d20', lw=2, ls='--', label='VIP = 1.0 (important)')
axes[0].axvline(0.8, color='orange',  lw=1, ls=':', label='VIP = 0.8 (moderate)')
axes[0].set_xlabel('VIP Score')
axes[0].set_title(f'Variable Importance in Projection\nPLS M={best_M_pls} — Bikeshare Demand')
axes[0].legend(fontsize=9)

# X loadings heatmap
n_show = min(best_M_pls, 4)
load_df = pd.DataFrame(pls_best.x_loadings_[:, :n_show],
                       index=features,
                       columns=[f'Comp {i+1}' for i in range(n_show)])
im = axes[1].imshow(load_df.values, cmap='RdYlGn', aspect='auto',
                    vmin=-load_df.values.__abs__().max(),
                    vmax=load_df.values.__abs__().max())
axes[1].set_xticks(range(n_show)); axes[1].set_xticklabels(load_df.columns)
axes[1].set_yticks(range(p)); axes[1].set_yticklabels(features, fontsize=9)
for i in range(p):
    for j in range(n_show):
        axes[1].text(j, i, f'{load_df.iloc[i,j]:.2f}',
                     ha='center', va='center', fontsize=8)
plt.colorbar(im, ax=axes[1], shrink=0.8)
axes[1].set_title('PLS X-Loadings\nRed = positive contribution, Green = negative')

plt.tight_layout()
plt.show()

important = vip_df[vip_df['VIP'] >= 1.0]['Feature'].tolist()
moderate  = vip_df[(vip_df['VIP'] >= 0.8) & (vip_df['VIP'] < 1.0)]['Feature'].tolist()
print(f"Features with VIP ≥ 1.0 (important for demand prediction):")
for _, row in vip_df[vip_df['VIP'] >= 1.0].iterrows():
    print(f"  {row.Feature:12s}: VIP = {row.VIP:.3f}")
print(f"\nFeatures with VIP 0.8–1.0 (moderate):")
for _, row in vip_df[(vip_df['VIP'] >= 0.8) & (vip_df['VIP'] < 1.0)].iterrows():
    print(f"  {row.Feature:12s}: VIP = {row.VIP:.3f}")
print()
print("📌 Policy implication: high-VIP features are the levers for demand.")
print("   Low-VIP features add noise without improving predictions.")
print("   A transport planner should focus monitoring on VIP ≥ 1 features.")


## 💼 Exercise

**Context:** The DC transport department wants a simple, explainable model for demand forecasting.

**Task 1 — Parsimonious model:** Using only the features with VIP ≥ 1.0, refit PLS(M=1). Compare test RMSE to the full PLS model. Is the simpler model good enough for operational planning?

**Task 2 — Seasonal VIP:** Split the dataset by `season` and compute VIP scores separately for winter vs summer. Which features matter more in winter? Which in summer? What does this suggest for seasonal infrastructure decisions?

**Task 3 — Forecast next hour:** Using the last row of the test set, predict the next hour's demand with all five models (OLS, Ridge, Lasso, PCR, PLS). Do they agree? What range would you report to city planners?

In [ ]:
# ── Exercise workspace ───────────────────────────────────────────────────────
# Variables: bikeshare, X_tr_s, X_te_s, y_tr, y_te, features, p
#            pls_best, pcr_best, ols, ridge, lasso
#            vip_df, best_M_pls, best_M_pcr

# Task 1 — Parsimonious model with only VIP ≥ 1.0 features
important_features = vip_df[vip_df['VIP'] >= 1.0]['Feature'].tolist()
imp_idx = [features.index(f) for f in important_features]

pls_simple = PLSRegression(n_components=1).fit(X_tr_s[:,imp_idx], y_tr)
mse_simple = mean_squared_error(y_te, pls_simple.predict(X_te_s[:,imp_idx]))
print(f"Full PLS (M={best_M_pls}, {p} features): RMSE = {pls_test_mse**0.5:.2f}")
print(f"Parsimonious PLS (M=1, {len(imp_idx)} features): RMSE = {mse_simple**0.5:.2f}")
print()

# Task 3 — Forecast for a single point
last_point = X_te_s[-1:, :]
print(f"Prediction for test observation {len(X_te_s)-1}:")
print(f"  Actual: {y_te[-1]:.0f} rentals/hr")
for name, model in [('OLS', ols), ('Ridge', ridge), ('Lasso', lasso),
                     (f'PCR M={best_M_pcr}', pcr_best),
                     (f'PLS M={best_M_pls}', pls_best)]:
    pred = float(model.predict(last_point).ravel()[0])
    print(f"  {name:15s}: {pred:.0f} rentals/hr")


In [ ]:
# @title 📝 Quiz — Partial Least Squares
# @markdown Answer each question, then run this cell.

# @markdown **Q1:** What does PLS optimise when constructing components, and how does this differ from PCA?
# @markdown **Q2:** Why does PLS typically need fewer components than PCR to achieve the same prediction error?
# @markdown **Q3:** What is a VIP score and what threshold indicates an important feature?
# @markdown **Q4:** In the Bikeshare context, why might `hr` (hour of day) have a high VIP score?
# @markdown **Q5:** A colleague says "just use Lasso — it also does feature selection." Give one reason to prefer PLS over Lasso for this dataset.

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the submission cell for AI feedback")


In [ ]:
_NB_NAME_ = "Partial Least Squares"
# @title 📋 Quiz Submission
# @markdown **Click ▶ Run** → copy the output → paste into Gemini in this Colab session.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}
_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

if "answers" not in globals():
    print("Run the quiz cell above first, then re-run this cell.")
else:
    qa = "\n".join(f"Q{i}: {str(v).strip() or '(no answer)'}"
                    for i, (_, v) in enumerate(answers.items(), 1))
    print(f'Please grade my quiz answers for the "{_NB_TITLE}" notebook')
    print(f'from "Data Pattern Recognition for the Rest of Us" (based on ISLP).')
    if GITHUB_USERNAME.strip(): print(f"Student: @{GITHUB_USERNAME.strip()}")
    print(); print(qa); print()
    print("For each: CORRECT/PARTIAL/INCORRECT, 2-3 sentence explanation,")
    print("ideal answer, Part to revisit if wrong.")
    print("End with: Overall grade + short study plan.")


---
### 🐛 Found a bug or have a suggestion?
[Open a GitHub Issue](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues/new?template=bug_report.md)

---
*Data Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*